In [59]:
!pip install tabula-py pdfplumber

In [60]:
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import re
import pandas as pd
import tabula
import pdfplumber
from io import BytesIO

In [61]:
def scrape_court_pdfs_and_extract_data(url):
    """
    Scrapes court names and PDF links from the specified URL, downloads each PDF,
    extracts the data, and formats it into sentences.

    Parameters:
    - url (str): The URL of the website to scrape.

    Returns:
    - list: A list of sentences extracted from each PDF.
    """
    court_urls = []

    # Set up a session with retry strategy
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=0.3, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retries)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    try:
        response = session.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Specify the container class name where the court names and PDF links are located
        container_class_name = 'gen-list yes-bg padding-20 border-radius-medium box-list normal-font'

        container = soup.find('div', class_=container_class_name)

        if container:
            court_name_divs = container.find_all('div', class_='list-text')
            pdf_links = container.find_all('a', href=re.compile(r'\.pdf$'))

            if len(court_name_divs) == len(pdf_links):
                for court_name_div, pdf_link in zip(court_name_divs, pdf_links):
                    court_name = court_name_div.get_text(strip=True)
                    pdf_url = pdf_link['href']

                    # Ensure the PDF link is an absolute URL
                    if not pdf_url.startswith('http'):
                        pdf_url = requests.compat.urljoin(url, pdf_url)

                    # Add court name and PDF URL to the list
                    court_urls.append([court_name, pdf_url])
            else:
                print("The number of court names and PDF links do not match.")
        else:
            print(f"No container found with class name: {container_class_name}")

    except requests.exceptions.RequestException as e:
        print(f"Failed to retrieve the website. Error: {e}")

    return court_urls


In [65]:
table_positions = ["is working as positon of Permanent Judge in", "is working as positon of Additional Judge in", "has been Transfered from"]
positions_for_judge_list = ["are working as positon of Permanent Judge in", "are working as positon of Additional Judge in", "are Transfered from"]


def find_sno_col(columns):
    for col in columns:
        COL = col
        col = col.lower()
        if( ('no.' in col) or ('s.' in col) or ("sl." in col) ):
            return COL
    return columns[0]


def find_num_rows(df):
    """
    Identify the first occurrence of a number in the first column to determine 'num_rows'.
    """
    # find s.no header name
    s_no_col = find_sno_col(df.columns)

    # Iterate over the rows of the dataframe
    for index, row in df.iterrows():
        if pd.isna(row[s_no_col]):
            continue
        # Check if the value in the first column can be interpreted as a number
        try:
            # print(str(row[s_no_col]))
            # Attempt to convert the value to a float (this works for both integers and floats)
            float(str(row[s_no_col])[:-1])
            return index  # Return the index where the first number is found
        except ValueError:
            continue
    return -1  # Default to 0 if no number is found


def combine_rows_as_header(df, num_rows):
    """
    Combine the first 'num_rows' rows of the dataframe to form a multi-line header.
    """
    header = df.iloc[:num_rows].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=0)
    header = [' '.join(head.split()) for head in header]  # Convert the header to a single string
    # print(header)
    df.columns = header  # Set the new header
    return df.iloc[num_rows:].reset_index(drop=True), header  # Return the dataframe without the first 'num_rows' rows


def combine_rows(df):
    # Fill missing values with empty strings for easier combination
    df.fillna('', inplace=True)

    # Create a list to store the combined rows
    combined_rows = []

    # find s.no header name
    s_no_col = find_sno_col(df.columns)

    # Iterate over the rows of the dataframe
    for index, row in df.iterrows():
        if pd.isna(row[s_no_col]) or row[s_no_col] == '':
            # If there's no Sl. No., combine with the previous row
            if combined_rows:
                # Combine the values of the current row with the last row in combined_rows
                last_row = combined_rows[-1]
                combined_rows[-1] = [str(last_row[i]) + ' ' + str(row[i]).strip() for i in range(len(row))]
        else:
            # If Sl. No. is present, append the row as a new entry
            combined_rows.append(list(row))

    # Convert the combined rows back into a dataframe
    combined_df = pd.DataFrame(combined_rows, columns=df.columns)
    return combined_df


def make_sentence(df_preprocessed, position_ind, court_name):
    sentences = []
    judges_names = []
    df_preprocessed = df_preprocessed.drop(find_sno_col(df_preprocessed.columns), axis = 1)
    # print(df_preprocessed)
    header_preprocessed = list(df_preprocessed.columns)
    # print(header_preprocessed)
    for index, row in df_preprocessed.iterrows():
        sentence = ""
        for i_col in range(len(header_preprocessed)):
            row[i_col] = row[i_col].strip()
            if(row[i_col] == ""  or  row[i_col] == "--"):
                continue
            elif('name' in header_preprocessed[i_col].lower()):
                ind_space_in_name = row[i_col].index(" ")
                if(ind_space_in_name < 4):
                    row[i_col] = row[i_col][:ind_space_in_name] + row[i_col][ind_space_in_name+1:]
                judges_names.append(row[i_col])
                sentence += f"{str(header_preprocessed[i_col])}, {str(row[i_col])} {table_positions[position_ind]} {court_name}. "
            else:
                sentence += f"{str(header_preprocessed[i_col])} is {str(row[i_col])}. "
        sentence = sentence.lower()
        sentence  = sentence.replace("pmt.", "permanent")
        sentence  = sentence.replace("addl.", "additional")
        sentence  = sentence.replace("\\", "")
        sentence  = sentence.replace("\n", "")
        sentences.append(sentence)
        print(index, sentence)
    if(len(judges_names) == 0):
        sentence_judge_names = f"There are no judges who {positions_for_judge_list[position_ind]} found in {court_name}."
    else:
        sentence_judge_names = f"The list of names of {len(judges_names)} judges who {positions_for_judge_list[position_ind]} in {court_name} are {', '.join(judges_names)}."
    print(sentence_judge_names)
    sentences.append(sentence_judge_names)
    return sentences


# Function to extract and print the first three lines of the PDF
def extract_judges_strength(pdf_url_hc):
    # Fetch the PDF from the URL
    response = requests.get(pdf_url_hc)

    if response.status_code == 200:
        # Use BytesIO to read the PDF content from the response
        pdf_file = BytesIO(response.content)

        # Open the PDF with pdfplumber
        with pdfplumber.open(pdf_file) as pdf:
            first_page = pdf.pages[0]  # Get the first page
            text = first_page.extract_text()  # Extract the text

            if text:
                lines = text.split('\n')  # Split the text into lines
                first_three_lines = lines[:3]  # Get the first three lines
                sentence = f"{first_three_lines[0]}, the {first_three_lines[1]} has an {first_three_lines[2]}"
                sentence = sentence.lower()
                sentence = sentence.replace(":", " of")
                sentence  = sentence.replace("pmt.", "permanent judge strength of")
                sentence  = sentence.replace("addl.", "and additional judge strength of")
                sentence  = sentence.replace("[", "which is ").replace("]","")
                return sentence
            else:
                print("No text found on the first page.")
                return ""
    else:
        print(f"Failed to fetch the PDF. Status code: {response.status_code}")
        return ""


def extract_data_from_pdf(court_name, pdf_url):
    # Process each table
    high_court_judges = []
    table_no = -1
    prev_header = None
    # Extract all tables from the PDF
    tables = tabula.read_pdf(pdf_url, pages='all', multiple_tables=True, stream=True)
    high_court_judges.append(extract_judges_strength(pdf_url))
    print(high_court_judges[0])
    for i, table in enumerate(tables):
        cols = []
        for col in table.columns:
            if 'unnamed' in col.lower():
                col = " "
            col = col.replace('.1', ' ')
            col = col.replace('.2', ' ')
            col = col.strip()
            cols.append(col)
        check_sno = find_sno_col(cols)
        if(check_sno.strip() == ""):
            cols[0] = "S.No"
        # print(cols)
        # Create a DataFrame for the headers (column names) as the first row
        header_table = pd.DataFrame([cols], columns=table.columns)
        table = pd.concat([header_table, table], ignore_index=True)

        # print(table)

        # Dynamically find the number of header rows based on the first occurrence of a number
        num_rows = find_num_rows(table)

        # print(f"Detected num_rows (header length) for Table {i+1}: {num_rows}")

        if(num_rows==-1):
            table_no += 1
            continue

        # Combine the detected rows as the header
        if num_rows > 0:
            table_no += 1
            header_processed_table, prev_header = combine_rows_as_header(table, num_rows)
        else:
            if(prev_header is not None):
                table.columns = prev_header[:len(table.columns)]
            header_processed_table = table  # No header rows found, leave table as-is
        preprocessed_table = combine_rows(header_processed_table)

        # print(table_positions[table_no])
        high_court_sentences = make_sentence(preprocessed_table, table_no, court_name)
        high_court_judges.extend(high_court_sentences)

    return high_court_judges


In [66]:
web_url = 'https://doj.gov.in/list-of-high-court-judges/'  # Replace with the actual URL
high_court_urls = scrape_court_pdfs_and_extract_data(web_url)
for court_name, pdf_url in high_court_urls:
    print(f"Court Name: {court_name}, PDF URL: {pdf_url}")
print(len(high_court_urls))

Court Name: ALLAHABAD HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/2024090222749154.pdf
Court Name: ANDHRA PRADESH HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/202409021136701139.pdf
Court Name: BOMBAY HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/202409021909604427.pdf
Court Name: CALCUTTA HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/20240902212923042.pdf
Court Name: CHHATTISGARH HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/20240902584825171.pdf
Court Name: DELHI HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/202409021529555779.pdf
Court Name: GAUHATI HIGH COURT, PDF URL: https://cdnbbsr.s3waas.gov.in/s35d6646aad9bcc0be55b2c82f69750387/uploads/2024/09/

In [67]:
high_courts_judges = []
for court_name, pdf_url in high_court_urls:
    print(f"Processing: {court_name}")
    high_courts_judges.extend(extract_data_from_pdf(court_name, pdf_url))

Processing: ALLAHABAD HIGH COURT


Sep 16, 2024 10:52:52 AM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>
Sep 16, 2024 10:52:56 AM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>
Sep 16, 2024 10:52:56 AM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>
Sep 16, 2024 10:53:00 AM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>
Sep 16, 2024 10:53:00 AM org.apache.pdfbox.pdmodel.font.PDTrueTypeFont <init>



as on 01/09/2024, the allahabad high court has an approved judge strength of 160 which is permanent judge strength of 119 and additional judge strength of 41
0 name of the judge s/shri justice, arun bhansali is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 08/01/2013. date of appointment as permanent judge is 07/01/2015. date of retirement is 14/10/2029. remarks is cj w.e.f. 05.02.2024 (phc: rajasthan). 
1 name of the judge s/shri justice, manoj kumar gupta is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 12/04/2013. date of appointment as permanent judge is 10/04/2015. date of retirement is 08/10/2026. 
2 name of the judge s/shri justice, anjani kumar mishra is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 12/04/2013. date of appointment as permanent judge is 10/0

<ipython-input-65-27c4be28446e>:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  combined_rows[-1] = [str(last_row[i]) + ' ' + str(row[i]).strip() for i in range(len(row))]
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446

as on 01/09/2024, the andhra pradesh high court has an approved judge strength of 37 which is permanent judge strength of of 28 and additional judge strength of of 09
0 name of the judge s/shri justice, dhiraj singh thakur is working as positon of permanent judge in andhra pradesh high court. source is bar. date of appointment as permanent judge is 08/03/2013. date of retirement is 24/04/2026. remarks is cj w.e.f. 28.07.23 (phc:j&k and ladakh]. 
1 name of the judge s/shri justice, guhanathan narendar is working as positon of permanent judge in andhra pradesh high court. source is bar. date of appointment as additional judge is 02/01/2015. date of appointment as permanent judge is 30/12/2017. date of retirement is 09/01/2026. remarks is joined on 30/10/23 (phc: karnataka). 
2 name of the judge s/shri justice, ravi nath tilhari is working as positon of permanent judge in andhra pradesh high court. source is bar. date of appointment as additional judge is 12/12/2019. date of appointment a

...

<ipython-input-65-27c4be28446e>:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  combined_rows[-1] = [str(last_row[i]) + ' ' + str(row[i]).strip() for i in range(len(row))]
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446

as on 01/09/2024, the tripura high court has an approved judge strength of 05 which is  permanent judge strength of of 04 and additional judge strength of of 01
0 name of the judge s/shri justice, aparesh kumar singh is working as positon of permanent judge in tripura high court. source is bar. date of appointment as additional judge is 24/01/2012. date of appointment as permanent judge is 16/01/2014. date of retirement is 06/07/2027. remarks is cj w.e.f. 17.04.2023 [phc: jharkhand]. 
1 name of the judge s/shri justice, todupunuri amarnath goud is working as positon of permanent judge in tripura high court. source is bar. date of appointment as additional judge is -. date of appointment as permanent judge is 21/09/2017. date of retirement is 28/02/2027. remarks is joined w.e.f. 28.10.2021 (phc: telangana). 
2 name of the judge s/shri justice, arindam lodh is working as positon of permanent judge in tripura high court. source is bar. date of appointment as permanent judge is 07/05/2018.

<ipython-input-65-27c4be28446e>:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  combined_rows[-1] = [str(last_row[i]) + ' ' + str(row[i]).strip() for i in range(len(row))]
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446

as on 01/09/2024, the uttarakhand high court has an approved judge strength of 11which is  permanent judge strength of of 09 and additional judge strength of of 02
0 name of the judge s/shri justice, kumari ritu bahri [h] is working as positon of permanent judge in uttarakhand high court. source is bar. date of appointment as additional judge is 16/08/2010. date of appointment as permanent judge is 23/02/2012. date of retirement/ date of occurrence of vacancy is 10/10/2024. remarks is cj w.e.f. 04/02/2024 (phc: p&h). 
1 name of the judge s/shri justice, manoj kumar tiwari is working as positon of permanent judge in uttarakhand high court. source is bar. date of appointment as additional judge is -. date of appointment as permanent judge is 19/05/2017. date of retirement/ date of occurrence of vacancy is 18/09/2027. 
2 name of the judge s/shri justice, ravindra maithani is working as positon of permanent judge in uttarakhand high court. source is service. date of appointment as permanen

<ipython-input-65-27c4be28446e>:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  combined_rows[-1] = [str(last_row[i]) + ' ' + str(row[i]).strip() for i in range(len(row))]
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446e>:84: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  row[i_col] = row[i_col].strip()
<ipython-input-65-27c4be28446

In [68]:
for i, sentence in enumerate(high_courts_judges):
    print(i, sentence)

0 as on 01/09/2024, the allahabad high court has an approved judge strength of 160 which is permanent judge strength of 119 and additional judge strength of 41
1 name of the judge s/shri justice, arun bhansali is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 08/01/2013. date of appointment as permanent judge is 07/01/2015. date of retirement is 14/10/2029. remarks is cj w.e.f. 05.02.2024 (phc: rajasthan). 
2 name of the judge s/shri justice, manoj kumar gupta is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 12/04/2013. date of appointment as permanent judge is 10/04/2015. date of retirement is 08/10/2026. 
3 name of the judge s/shri justice, anjani kumar mishra is working as positon of permanent judge in allahabad high court. source is bar. date of appointment as additional judge is 12/04/2013. date of appointment as permanent judge is 10

In [69]:
len(high_courts_judges)

898

In [42]:
tabula.read_pdf?